# BabyLM BLiMP Benchmark

This notebook follows the mini-project instructions for **Task 1: benchmark the base BabyLM**.

Evaluation setup:
- Model: `babylm/babyllama-100m-2024`
- Data: `../../data/blimp/blimp_validation.json`
- Metric: pairwise grammaticality judgment accuracy
- Scoring rule: choose the sentence with the higher sentence log probability
- Important: **no length normalization**

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

import nltk
import pandas as pd
import torch
import torch.nn.functional as F
from nltk.tokenize import word_tokenize
from transformers import AutoModelForCausalLM, AutoTokenizer

c:\Users\PC\miniconda3\envs\cogsci\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
for resource in ("punkt", "punkt_tab"):
    try:
        nltk.data.find(f"tokenizers/{resource}")
    except LookupError:
        nltk.download(resource)

In [ ]:
DATA_PATH = Path("../../data/blimp/blimp_validation.json")
MODEL_NAME = "babylm/opt-125m-strict-2023"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with DATA_PATH.open(encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} BLiMP minimal pairs")
print(f"Running on: {DEVICE}")
data[0]

Loaded 6700 BLiMP minimal pairs
Running on: cpu


{'sentence_good': "Stacy's associate hasn't passed a lot of grocery stores and the girls' doctor hasn't passed a few white grocery stores.",
 'sentence_bad': "Stacy's associate hasn't passed a lot of dirty grocery stores and the girls' doctor hasn't passed a few white.",
 'field': 'syntax',
 'linguistics_term': 'ellipsis',
 'UID': 'ellipsis_n_bar_2',
 'simple_LM_method': True,
 'one_prefix_method': False,
 'two_prefix_method': False,
 'lexically_identical': False,
 'pairID': '248'}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_orig = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model_orig.to(DEVICE)
model_orig.eval();

Loading weights: 100%|██████████| 147/147 [00:00<00:00, 1509.65it/s]


In [ ]:
@torch.inference_mode()
def calculate_sentence_log_probabilities(model, tokenizer, sentences, device=DEVICE):
    encoded = tokenizer(
        sentences,
        return_tensors="pt",
        padding=True,
        truncation=False,
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :]
    target_ids = input_ids[:, 1:]
    target_mask = attention_mask[:, 1:]

    token_log_probs = F.log_softmax(logits, dim=-1)
    next_token_log_probs = token_log_probs.gather(
        dim=-1,
        index=target_ids.unsqueeze(-1),
    ).squeeze(-1)

    sentence_log_probs = (next_token_log_probs * target_mask).sum(dim=1)
    return sentence_log_probs.detach().cpu().tolist()


def calculate_sentence_probability(model, tokenizer, sentence):
    return calculate_sentence_log_probabilities(model, tokenizer, [sentence])[0]


def evaluate_blimp(model, tokenizer, dataset, batch_size=16, device=DEVICE):
    pair_results = []
    term_stats = defaultdict(lambda: {"correct": 0, "total": 0})

    for start in range(0, len(dataset), batch_size):
        batch = dataset[start:start + batch_size]
        good_sentences = [item["sentence_good"] for item in batch]
        bad_sentences = [item["sentence_bad"] for item in batch]

        good_scores = calculate_sentence_log_probabilities(
            model, tokenizer, good_sentences, device=device
        )
        bad_scores = calculate_sentence_log_probabilities(
            model, tokenizer, bad_sentences, device=device
        )

        for item, good_score, bad_score in zip(batch, good_scores, bad_scores):
            correct = good_score > bad_score
            pair_results.append({
                "linguistics_term": item["linguistics_term"],
                "UID": item["UID"],
                "pairID": item["pairID"],
                "good_score": good_score,
                "bad_score": bad_score,
                "correct": correct,
            })

            term_stats[item["linguistics_term"]]["correct"] += int(correct)
            term_stats[item["linguistics_term"]]["total"] += 1

    overall_accuracy = sum(result["correct"] for result in pair_results) / len(pair_results)
    per_term = pd.DataFrame([
        {
            "linguistics_term": term,
            "accuracy": stats["correct"] / stats["total"],
            "num_pairs": stats["total"],
        }
        for term, stats in sorted(term_stats.items())
    ]).sort_values("accuracy", ascending=False).reset_index(drop=True)

    return overall_accuracy, per_term, pd.DataFrame(pair_results)


In [ ]:
overall_accuracy, per_term_results, pair_results = evaluate_blimp(
    model_orig,
    tokenizer,
    data,
    batch_size=8,
    device=DEVICE,
)

print(f"Overall BLiMP accuracy: {overall_accuracy:.4f}")
per_term_results

Overall BLiMP accuracy: 0.7593


,linguistics_term,accuracy,num_pairs
0,anaphor_agreement,0.975000,200
1,determiner_noun_agreement,0.923750,800
2,irregular_forms,0.895000,200
3,ellipsis,0.835000,200
4,subject_verb_agreement,0.825000,600
5,control_raising,0.768000,500
6,argument_structure,0.764444,900
7,filler_gap_dependency,0.748571,700
8,binding,0.737143,700
9,quantifiers,0.720000,400


In [ ]:
example_index = 1

good_sentence = data[example_index]["sentence_good"]
bad_sentence = data[example_index]["sentence_bad"]

good_log_prob = calculate_sentence_probability(model_orig, tokenizer, good_sentence)
bad_log_prob = calculate_sentence_probability(model_orig, tokenizer, bad_sentence)

print("Good sentence:")
print(good_sentence)
print(f"Log probability: {good_log_prob:.4f}")
print()
print("Bad sentence:")
print(bad_sentence)
print(f"Log probability: {bad_log_prob:.4f}")
print()
print("Model chooses grammatical sentence:", good_log_prob > bad_log_prob)

Good sentence:
Timothy's senator isn't questioning one cousin of and Renee isn't questioning a lot employed cousin of.
Log probability: -135.2633

Bad sentence:
Timothy's senator isn't questioning one excited cousin of and Renee isn't questioning a lot employed.
Log probability: -131.1988

Model chooses grammatical sentence: False


## Official Word Count Method

The slides specify a word-based token count using the NLTK tokenizer for the fine-tuning data budget.

In [ ]:
def count_word(text):
    tokens = word_tokenize(text)
    return len(tokens)

In [ ]:
text = "NLTK makes processing text easy! Don't you think so?"
count_word(text)

12

In [ ]:
for item in data:
    item["sentence_good_word_count"] = count_word(item["sentence_good"])
    item["sentence_bad_word_count"] = count_word(item["sentence_bad"])

word_count_summary = {
    "num_pairs": len(data),
    "total_sentence_good_words": sum(item["sentence_good_word_count"] for item in data),
    "total_sentence_bad_words": sum(item["sentence_bad_word_count"] for item in data),
    "average_sentence_good_words": sum(item["sentence_good_word_count"] for item in data) / len(data),
    "average_sentence_bad_words": sum(item["sentence_bad_word_count"] for item in data) / len(data),
    "average_total_words_per_pair": sum(
        item["sentence_good_word_count"] + item["sentence_bad_word_count"] for item in data
    ) / len(data),
}

word_count_summary

{'num_pairs': 6700,
 'total_sentence_good_words': 59241,
 'total_sentence_bad_words': 59397,
 'average_sentence_good_words': 8.841940298507463,
 'average_sentence_bad_words': 8.865223880597014,
 'average_total_words_per_pair': 17.707164179104478}

In [ ]:
benchmark_summary = {
    "model_name": MODEL_NAME,
    "dataset_size": len(data),
    "overall_accuracy": overall_accuracy,
    "per_term_accuracy": {
        row.linguistics_term: row.accuracy
        for row in per_term_results.itertuples(index=False)
    },
}

with open("../../data/reports/blimp_benchmark/opt-125m-strict-2023_benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump(benchmark_summary, f, indent=2)

benchmark_summary

{'model_name': 'babylm/babyllama-100m-2024',
 'dataset_size': 6700,
 'overall_accuracy': 0.7592537313432836,
 'per_term_accuracy': {'anaphor_agreement': 0.975,
  'determiner_noun_agreement': 0.92375,
  'irregular_forms': 0.895,
  'ellipsis': 0.835,
  'subject_verb_agreement': 0.825,
  'control_raising': 0.768,
  'argument_structure': 0.7644444444444445,
  'filler_gap_dependency': 0.7485714285714286,
  'binding': 0.7371428571428571,
  'quantifiers': 0.72,
  'npi_licensing': 0.68,
  'island_effects': 0.545}}